# DetectAI — Fine-tuning BERT to detect AI-generated text
**An Explainable BERT-Based System for Detecting AI-Generated Text**
MSc Data Science & Analytics · 3rd semester NLP · Jain University
Team: **Khushi S, Akhila Vaidya, Kalpana**

This notebook is **Phase 2** of DetectAI. Phase 1 (on our laptop) built the dataset and trained 5 classical baselines
(Naive Bayes, Logistic Regression, SVM, stylometric). Here we fine-tune **`bert-base-uncased`**, a Transformer,
on the same data and the same train/val/test split, compare it to the baselines, test its robustness, and **explain** its decisions.

**How to run (≈ 25–35 min on a free T4):**
1. `Runtime → Change runtime type → T4 GPU → Save`
2. `Runtime → Run all`
3. When the upload button appears, upload the 5 files listed in step 3.
4. At the end, `detectai_bert_outputs.zip` downloads automatically (model + metrics + figures).

| Rubric item | Where in this notebook |
|---|---|
| NLP Concepts & Methodology | WordPiece tokenization, Transformers, attention, contextual embeddings |
| Implementation / Coding | Hugging Face `Trainer` fine-tuning |
| Dataset Preparation & Preprocessing | data loading, split, tokenization, max length choice |
| Results, Analysis & Evaluation | metrics, confusion matrix, ROC, baseline comparison, per-domain, robustness, error analysis |
| Explainability / Ethics | Gradient×Input saliency, attention analysis + caveat, false-positive discussion |

In [ ]:
# Step 1 - Install pinned library versions.
# Colab already has PyTorch with GPU (CUDA) support, so we do NOT reinstall torch.
# These are the same versions as our local demo backend, so the saved model loads there too.
import sys
if "google.colab" in sys.modules:
    !pip install -q transformers==5.19.0 accelerate==1.15.0 nltk==3.10.3

**What this does:** installs the exact library versions we use. `transformers` (by Hugging Face) gives us BERT and the
training loop; `accelerate` runs training on the GPU; `nltk` is needed for the text-editing robustness test.
Pinning versions means the notebook behaves the same every time, and the model we save here will load in our local demo.

**Syllabus link — Unit 1: NLP libraries** (NLTK and Hugging Face Transformers).

In [ ]:
# Step 2 - Check that a GPU is available
import torch

print("PyTorch version:", torch.__version__)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}  |  memory: {gpu.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU found. Go to Runtime -> Change runtime type -> T4 GPU, then run again.")

**What this does:** checks that Colab gave us a GPU (it should say *Tesla T4*). BERT has 110 million parameters;
training it on a CPU would take many hours, on a T4 GPU it takes about 20–30 minutes.

**Concept:** deep-learning models like Transformers need parallel hardware (GPUs) for practical training.

In [ ]:
# Step 3 - Imports, settings and figure style
import json, os, random, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback,
                          set_seed)

# SMOKE_TEST is only used by us to check the notebook quickly on a laptop (tiny model, tiny data).
# On Colab it is False, so the real bert-base-uncased model is trained on all the data.
SMOKE_TEST = os.environ.get("DETECTAI_SMOKE") == "1"

MODEL_NAME = "bert-base-uncased"
TOKENIZER_NAME = "bert-base-uncased"
if SMOKE_TEST:   # laptop check only: a tiny, untrained 2-layer BERT so every cell runs in minutes
    from transformers import BertConfig, BertForSequenceClassification
    BertForSequenceClassification(BertConfig(hidden_size=64, num_hidden_layers=2, num_attention_heads=2,
                                             intermediate_size=128)).save_pretrained("tiny_bert_smoke")
    MODEL_NAME = "tiny_bert_smoke"
MAX_LEN = 320          # max WordPiece tokens per text (justified in step 5)
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
MAX_EPOCHS = 4         # early stopping may stop sooner
WARMUP = 0.1           # first 10% of steps: learning rate rises slowly from 0
WEIGHT_DECAY = 0.01
SEED = 42
set_seed(SEED)

OUT_DIR = Path("detectai_bert_outputs")
FIG_DIR = OUT_DIR / "figures"
MODEL_DIR = OUT_DIR / "detectai_bert_model"
FIG_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# One shared look for all figures (same colours as our Phase 1 figures)
HUMAN_COLOR, AI_COLOR = "#2a78d6", "#eb6834"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MUTED = "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight", "font.size": 10,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": "#e4e3df",
    "axes.axisbelow": True, "legend.frameon": False, "lines.linewidth": 2,
})
print("Model:", MODEL_NAME, "| device:", device, "| smoke test:", SMOKE_TEST)

**What this does:** loads the libraries and sets every training setting in one place:
learning rate 2e-5, batch size 16, up to 4 epochs, 10% warm-up, weight decay 0.01, max length 320 tokens.
A fixed **random seed (42)** makes results repeatable.

**Concept — fine-tuning hyper-parameters:** a small learning rate (2e-5) gently adjusts BERT's pre-trained knowledge
instead of overwriting it; *warm-up* avoids large, destabilising updates at the start; *weight decay* is a
regularisation that discourages over-fitting.

In [ ]:
# Step 4 - Upload the data files from our laptop
REQUIRED = ["detectai_dataset.csv", "train_augmented.csv", "baseline_metrics.json",
            "robustness.json", "augment.py"]

missing = [f for f in REQUIRED if not Path(f).exists()]
if missing:
    try:
        from google.colab import files
        print("Please upload:", ", ".join(missing))
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Put these files next to the notebook: {missing}")

missing = [f for f in REQUIRED if not Path(f).exists()]
assert not missing, f"Still missing: {missing}"
print("All files present:", ", ".join(REQUIRED))

**What this does:** opens an upload button (only if the files are not already there). We upload 5 files from Phase 1:

| File | What it is |
|---|---|
| `detectai_dataset.csv` | 8,895 texts (human vs ChatGPT) with a fixed `split` column (train/val/test) |
| `train_augmented.csv` | lightly edited copies of the AI **training** texts |
| `baseline_metrics.json` | results of our 5 classical models, for comparison |
| `robustness.json` | classical models' robustness results, for comparison |
| `augment.py` | the exact text-editing code, so the robustness test edits text the same way |

**Concept — reproducibility:** BERT is evaluated on **exactly the same test texts** as the baselines, so the comparison is fair.

In [ ]:
# Step 5 - Load the data and respect the fixed train / val / test split
df = pd.read_csv("detectai_dataset.csv")
aug = pd.read_csv("train_augmented.csv")

# Safety check: we must use the CURRENT, cleaned dataset (Reddit markdown such as *emphasis*,
# "(URL_0)" placeholders removed). An old file would let the model learn "asterisks = human".
leftover = int(df["text"].str.contains(r"\*|URL_\d", regex=True).sum()
               + aug["text"].str.contains(r"\*|URL_\d", regex=True).sum())
assert leftover == 0, (f"{leftover} texts still contain markdown - please upload the LATEST "
                       "detectai_dataset.csv and train_augmented.csv from the colab_upload folder")
print(f"Dataset check passed: {len(df)} cleaned texts (no markdown artefacts)")

train_df = pd.concat([df[df["split"] == "train"], aug], ignore_index=True)   # augmented copies: TRAIN only
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)                  # original order kept

if SMOKE_TEST:   # tiny subset for a quick laptop check
    train_df = train_df.sample(240, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(64, random_state=SEED).reset_index(drop=True)
    test_df = test_df.sample(80, random_state=SEED).sort_index().reset_index(drop=True)

summary = pd.DataFrame({
    "texts": [len(train_df), len(val_df), len(test_df)],
    "human": [(d["label"] == 0).sum() for d in (train_df, val_df, test_df)],
    "AI": [(d["label"] == 1).sum() for d in (train_df, val_df, test_df)],
}, index=["train (+augmented)", "validation", "test"])
print(summary, "\n")
print("Test texts per domain:\n", test_df["domain"].value_counts().to_string())

**What this does:** reads the dataset, checks that it is the latest **cleaned** version (no Reddit markdown like `*word*`
or `(URL_0)`, which appeared almost only in human texts and would be an unfair shortcut), and splits it using the **fixed `split` column** made in Phase 1
(70% train / 15% validation / 15% test, stratified by label and domain). The edited AI copies are added to
**training only**; validation and test are untouched real texts.

**Concept — dataset preparation (rubric item 4):** the *validation* set is used to pick the best epoch (early stopping);
the *test* set is used only once at the end, so the final score is an honest estimate on unseen text.

In [ ]:
# Step 6 - WordPiece tokenization demo + choosing the maximum length
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

demo_sentences = [
    "Unbelievably, the researchers' findings were groundbreaking!",
    "In today's rapidly evolving landscape, it is crucial to delve into this.",
]
for s in demo_sentences:
    enc = tokenizer(s)
    print("Text      :", s)
    print("WordPieces:", tokenizer.convert_ids_to_tokens(enc["input_ids"]))
    print("Token IDs :", enc["input_ids"], "\n")

# How many WordPiece tokens do our texts have? (a harmless "longer than 512" warning may appear: we only count here)
lengths = np.array([len(ids) for ids in tokenizer(df["text"].tolist(), truncation=False)["input_ids"]])
print(f"Tokens per text: median {np.median(lengths):.0f}, 95th percentile {np.percentile(lengths, 95):.0f}, "
      f"max {lengths.max()}")
print(f"Texts longer than MAX_LEN={MAX_LEN}: {(lengths > MAX_LEN).mean() * 100:.1f}% (only these get truncated)")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(lengths, bins=50, color=HUMAN_COLOR)
ax.axvline(MAX_LEN, color=AI_COLOR, linestyle="--", label=f"MAX_LEN = {MAX_LEN}")
ax.set_xlabel("WordPiece tokens per text"); ax.set_ylabel("Number of texts")
ax.set_title("Text length in WordPiece tokens"); ax.legend()
fig.savefig(FIG_DIR / "wordpiece_length_distribution.png"); plt.show()

**What this does:** shows how BERT "reads" text. BERT does not see whole words — it uses **WordPiece tokenization**,
which splits rare words into known sub-word pieces (pieces starting with `##` continue the previous word). It also adds
two special tokens: **`[CLS]`** at the start (its final vector is used for classification) and **`[SEP]`** at the end.
Every piece is then mapped to a number (token ID).

We then count tokens for every text (printed above). Our texts are 20–300 words, which is roughly 25–500 tokens.
**MAX_LEN = 320** keeps most texts completely; the longer ones keep their first 320 tokens (≈ 240 words), which is still plenty of
evidence. A longer limit would make training much slower on a free GPU, because attention cost grows with the square of the length.

**Syllabus links — Unit 1: tokenization and morphology.** "Unbelievably" becomes `un ##bel ##ie ##va ##bly`: the prefix *un-* is
kept as its own piece (a real morpheme), but the rest is split by frequency, not by grammar — WordPiece is a statistical
approximation of morphology. One vocabulary of ~30k pieces can represent any word, even ones never seen in training.

In [ ]:
# Step 7 - Turn texts into BERT inputs
class TextDataset(torch.utils.data.Dataset):
    """Tokenizes all texts once; returns one example (token IDs + attention mask + label) at a time."""
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = None if labels is None else list(labels)

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:          # no labels when we only want predictions
            item["labels"] = int(self.labels[i])
        return item

train_ds = TextDataset(train_df["text"], train_df["label"])
val_ds = TextDataset(val_df["text"], val_df["label"])
test_ds = TextDataset(test_df["text"], test_df["label"])

# Pads each batch only up to its longest text (faster than always padding to 320)
collator = DataCollatorWithPadding(tokenizer)
print(f"Encoded: {len(train_ds)} train, {len(val_ds)} val, {len(test_ds)} test examples")

**What this does:** converts every text into the three things BERT needs: **input IDs** (the WordPiece numbers),
an **attention mask** (1 = real token, 0 = padding) and the **label** (0 = human, 1 = AI).
Texts longer than 320 tokens are truncated. Each batch is padded only to the length of its longest text, which saves GPU time.

Note that, unlike our classical models, we **keep stopwords and punctuation** and do no stemming or lemmatization —
the only normalisation is lowercasing (that is what *uncased* means). BERT uses every word as context.

**Syllabus link — Unit 2: text preprocessing for deep learning** (contrast with stopword removal / lemmatization for TF-IDF).

In [ ]:
# Step 8 - Fine-tune BERT with the Hugging Face Trainer
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label={0: "human", 1: "AI"}, label2id={"human": 0, "AI": 1})

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {"accuracy": accuracy_score(labels, preds), "precision": precision_score(labels, preds, zero_division=0),
            "recall": recall_score(labels, preds), "f1": f1_score(labels, preds),
            "roc_auc": roc_auc_score(labels, probs)}

args = TrainingArguments(
    output_dir="checkpoints",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    num_train_epochs=MAX_EPOCHS,
    warmup_steps=WARMUP,                 # a value below 1 means "this fraction of all steps"
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",               # evaluate on the validation set after every epoch
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=5 if SMOKE_TEST else 50,
    load_best_model_at_end=True,         # keep the epoch with the best validation F1
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),      # mixed precision: faster on the T4 GPU
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],  # stop if val F1 does not improve
)

t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60
print(f"Training finished in {train_minutes:.1f} minutes. Best validation F1: {trainer.state.best_metric:.4f}")

**What this does:** loads pre-trained BERT and adds a small classification layer on top of the `[CLS]` vector,
then trains the whole network on our labelled texts. After every epoch it measures F1 on the validation set and keeps
the best epoch; if F1 stops improving it stops early (**early stopping**), which prevents over-fitting.
`fp16` (mixed precision) roughly doubles speed on the T4.

**Syllabus links — Unit 4: Transformers and attention.** BERT is a 12-layer Transformer *encoder*. In every layer,
**self-attention** lets each word look at every other word in the text, so each word's vector depends on its context.
**Unit 2: contextual embeddings (BERT)** — unlike TF-IDF or Word2Vec, the same word gets a different vector in different sentences.
This is **transfer learning**: BERT already learned English from 3.3 billion words; we only teach it our task.

In [ ]:
# Step 9 - Training curves
history = trainer.state.log_history
train_log = pd.DataFrame([h for h in history if "loss" in h and "eval_loss" not in h])
eval_log = pd.DataFrame([h for h in history if "eval_loss" in h])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
if len(train_log):
    axes[0].plot(train_log["epoch"], train_log["loss"], color=HUMAN_COLOR, label="training loss")
axes[0].plot(eval_log["epoch"], eval_log["eval_loss"], color=AI_COLOR, marker="o", markersize=8,
             label="validation loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-entropy loss"); axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(eval_log["epoch"], eval_log["eval_f1"], color=SERIES[2], marker="o", markersize=8, label="validation F1")
axes[1].plot(eval_log["epoch"], eval_log["eval_accuracy"], color=SERIES[6], marker="o", markersize=8,
             label="validation accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_title("Validation F1 and accuracy"); axes[1].legend()
fig.savefig(FIG_DIR / "bert_training_curves.png"); plt.show()

epoch_table = eval_log[["epoch", "eval_loss", "eval_accuracy", "eval_f1", "eval_roc_auc"]].round(4)
print(epoch_table.to_string(index=False))
best_epoch = float(eval_log.loc[eval_log["eval_f1"].idxmax(), "epoch"])
print("Best epoch (kept):", best_epoch)

**What this does:** plots how the **loss** (the model's error) changed during training, and the validation F1 / accuracy
after each epoch. Training loss should fall. If validation loss starts rising while training loss keeps falling, the model is
starting to **over-fit** (memorising training texts) — that is exactly when early stopping keeps the earlier, better epoch.

**Concept — model evaluation during training (rubric item 5).**

In [ ]:
# Step 10 - Save the fine-tuned model and tokenizer
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
print("Saved to", MODEL_DIR, ":", sorted(p.name for p in MODEL_DIR.iterdir()))

**What this does:** saves the best fine-tuned BERT model and its tokenizer. Our local demo backend will load this folder to make
live predictions. We save it now (not only at the end) because the attention analysis in step 15 reloads it.

In [ ]:
# Step 11 - Final evaluation on the untouched TEST set
def predict_proba(texts, batch_size=64):
    """Probability that each text is AI-generated."""
    ds = TextDataset(texts)
    logits = trainer.predict(ds).predictions
    return torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()

y_test = test_df["label"].values
test_probs = predict_proba(test_df["text"].tolist())
test_preds = (test_probs >= 0.5).astype(int)
test_df["prob_ai"], test_df["pred"] = test_probs, test_preds

def all_metrics(y, prob, domains):
    pred = (prob >= 0.5).astype(int)
    return {
        "accuracy": round(accuracy_score(y, pred), 4), "precision": round(precision_score(y, pred), 4),
        "recall": round(recall_score(y, pred), 4), "f1": round(f1_score(y, pred), 4),
        "roc_auc": round(roc_auc_score(y, prob), 4),
        "per_domain_accuracy": {d: round(accuracy_score(y[domains == d], pred[domains == d]), 4)
                                for d in sorted(set(domains))},
        "confusion_matrix": confusion_matrix(y, pred).tolist(),
    }

bert_test = all_metrics(y_test, test_probs, test_df["domain"].values)
for k in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    print(f"{k:10s}: {bert_test[k]:.4f}")

# Confusion matrix
cm = np.array(bert_test["confusion_matrix"])
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues"); ax.grid(False)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14,
                color="white" if cm[i, j] > cm.max() * 0.6 else "#0b0b0b")
ax.set_xticks([0, 1], ["human", "AI"]); ax.set_yticks([0, 1], ["human", "AI"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("BERT - confusion matrix (test)")
fig.savefig(FIG_DIR / "bert_confusion_matrix.png"); plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_test, test_probs)
fig, ax = plt.subplots(figsize=(4.8, 4.5))
ax.plot(fpr, tpr, color=SERIES[5], label=f"BERT (AUC {bert_test['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "--", color=MUTED, linewidth=1)
ax.set_xlabel("False positive rate (human flagged as AI)"); ax.set_ylabel("True positive rate (AI caught)")
ax.set_title("BERT - ROC curve (test)"); ax.legend(loc="lower right")
fig.savefig(FIG_DIR / "bert_roc_curve.png"); plt.show()

**What this does:** runs the final model once on the **test set** (texts it has never seen) and reports:
- **Accuracy** – share of all texts classified correctly.
- **Precision** – of the texts we flagged as AI, how many really were AI (low precision = innocent humans accused).
- **Recall** – of all AI texts, how many we caught.
- **F1** – the balance (harmonic mean) of precision and recall.
- **ROC-AUC** – how well the AI-probability ranks AI texts above human ones, across every threshold (1.0 = perfect, 0.5 = guessing).
- **Confusion matrix** – the four counts: humans correctly passed, humans wrongly flagged, AI missed, AI caught.

**Syllabus link — Unit 4: evaluation metrics** (accuracy, precision, recall, F1, confusion matrix).

In [ ]:
# Step 12 - Compare BERT with the 5 classical baselines (same test set)
baselines = json.load(open("baseline_metrics.json"))["models"]
all_models = {**baselines, "BERT (fine-tuned)": bert_test}

table = pd.DataFrame({name: {k: m[k] for k in ["accuracy", "precision", "recall", "f1", "roc_auc"]}
                      for name, m in all_models.items()}).T
print(table.round(4).to_string())

keys, labels = ["accuracy", "f1", "roc_auc"], ["Accuracy", "F1", "ROC-AUC"]
names = list(all_models)
x, w = np.arange(len(keys)), 0.8 / len(names)
fig, ax = plt.subplots(figsize=(10, 4.2))
for i, (name, color) in enumerate(zip(names, SERIES)):
    bars = ax.bar(x + (i - (len(names) - 1) / 2) * w, [all_models[name][k] for k in keys],
                  width=w * 0.92, color=color, label=name)
    if name.startswith("BERT"):
        ax.bar_label(bars, fmt="%.3f", fontsize=8, color=MUTED)
ax.set_xticks(x, labels); ax.set_ylim(0.5, 1.0)
ax.set_title("BERT vs classical baselines (test set)")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
fig.savefig(FIG_DIR / "bert_vs_baselines.png"); plt.show()

**What this does:** puts BERT next to our five classical models — **Bag-of-Words + Naive Bayes, TF-IDF + Logistic Regression
(with and without stopwords), character n-gram TF-IDF + SVM, and stylometric features + Logistic Regression** — all scored on the
identical test texts.

**Why BERT should do better:** Bag-of-Words and TF-IDF only count words and ignore order and meaning; BERT reads every word **in context**
(self-attention), so it can pick up the smooth, generic phrasing typical of ChatGPT even when individual words are ordinary.

**Syllabus links — Unit 2** (Bag of Words, TF-IDF vs contextual embeddings) and **Unit 4** (Naive Bayes, Logistic Regression, SVM vs Transformers).

In [ ]:
# Step 13 - Accuracy per domain
domains = sorted(test_df["domain"].unique())
compare = {"TF-IDF + LR (stopwords kept)": baselines["TF-IDF + LR (stopwords kept)"],
           "Char n-gram TF-IDF + SVM": baselines["Char n-gram TF-IDF + SVM"],
           "BERT (fine-tuned)": bert_test}
colors = [SERIES[2], SERIES[3], SERIES[5]]

x, w = np.arange(len(domains)), 0.8 / len(compare)
fig, ax = plt.subplots(figsize=(9, 4))
for i, ((name, m), color) in enumerate(zip(compare.items(), colors)):
    bars = ax.bar(x + (i - (len(compare) - 1) / 2) * w,
                  [m["per_domain_accuracy"].get(d, np.nan) for d in domains], width=w * 0.92, color=color, label=name)
    ax.bar_label(bars, fmt="%.2f", fontsize=7, color=MUTED)
ax.set_xticks(x, domains); ax.set_ylim(0.5, 1.02); ax.set_ylabel("Accuracy")
ax.set_title("Accuracy per domain (test set)"); ax.legend(fontsize=8, loc="lower left")
fig.savefig(FIG_DIR / "bert_per_domain_accuracy.png"); plt.show()

print(pd.DataFrame({n: m["per_domain_accuracy"] for n, m in compare.items()}).round(4).to_string())

**What this does:** breaks accuracy down by text type: **Q&A (HC3)** casual answers, student **essays**, **Reuters** news and
**wp** creative stories. A detector that only works on one kind of writing is not useful in practice, so this shows where each model
is strong or weak. In Phase 1 the Q&A domain (short, casual answers — closest to what a teacher would type live) was the hardest for
the classical models.

**Syllabus link — Unit 5: NLP applications** (the same detector must work across real-world text genres).

In [ ]:
# Step 14 - Robustness: does BERT still catch AI text after light human-style editing?
import nltk
# Colab reaches the internet through a proxy. Newer NLTK versions refuse to download through a
# proxy ("Security Violation") unless we explicitly allow it - so we switch on BOTH options
# (environment variable + pathsec flag) before any download.
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
try:
    import nltk.pathsec
    nltk.pathsec.ALLOW_PROXIED_FETCH = True
except (ImportError, AttributeError):
    pass   # older NLTK without this security check - nothing to switch on
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
from augment import edit_text   # the same edits used in Phase 1

rng = random.Random(7)          # same seed as Phase 1 robustness_test.py -> identical edited texts
edited = test_df["text"].copy()
ai_rows = test_df["label"] == 1
edited[ai_rows] = test_df.loc[ai_rows, "text"].apply(lambda t: edit_text(t, rng))

print("Example edit:\n BEFORE:", test_df.loc[ai_rows, "text"].iloc[0][:250], "\n AFTER :", edited[ai_rows].iloc[0][:250])

edited_probs = predict_proba(edited.tolist())
edited_preds = (edited_probs >= 0.5).astype(int)
bert_robust = {
    "accuracy_original": round(accuracy_score(y_test, test_preds), 4),
    "accuracy_edited": round(accuracy_score(y_test, edited_preds), 4),
    "ai_recall_original": round(recall_score(y_test, test_preds), 4),
    "ai_recall_edited": round(recall_score(y_test, edited_preds), 4),
}
bert_robust["accuracy_drop"] = round(bert_robust["accuracy_original"] - bert_robust["accuracy_edited"], 4)
print("\nBERT:", bert_robust)

robust = json.load(open("robustness.json"))["models"]
robust["BERT (fine-tuned)"] = bert_robust
names = list(robust)
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(10, 4))
b1 = ax.bar(x - 0.2, [robust[n]["accuracy_original"] for n in names], 0.38, color=HUMAN_COLOR, label="Original test set")
b2 = ax.bar(x + 0.2, [robust[n]["accuracy_edited"] for n in names], 0.38, color=AI_COLOR, label="AI texts lightly edited")
ax.bar_label(b1, fmt="%.3f", fontsize=7, color=MUTED); ax.bar_label(b2, fmt="%.3f", fontsize=7, color=MUTED)
ax.set_xticks(x, [n.replace(" (", "\n(").replace(" + ", " +\n", 1) for n in names], fontsize=8)
ax.set_ylim(0.4, 1.05); ax.set_ylabel("Accuracy"); ax.legend(loc="lower left", fontsize=8)
ax.set_title("Robustness to light editing of AI texts")
fig.savefig(FIG_DIR / "bert_robustness.png"); plt.show()

**What this does:** people often tweak ChatGPT output before submitting it. We simulate this on the **AI texts of the test set**:
swap ~15% of words for **WordNet synonyms**, **shuffle sentence order**, and **delete filler words** — then check how much accuracy drops.
Human texts are not changed. We use the identical edits (same code, same seed) as for the classical models.

*(Technical note: the first lines allow NLTK to download its WordNet/stopword data through Colab's network proxy,
which newer NLTK versions block by default.)*

**Syllabus links — Unit 2: word meaning / lexical resources (WordNet synonyms)**, **Unit 1: ambiguity** (a synonym picked without context
can change the meaning — e.g. "bank" → "depository"), and **Unit 5: ethical issues** (a detector that is easy to evade gives false confidence).

In [ ]:
# Step 15a - Explainability: Gradient x Input word-level saliency
from IPython.display import HTML, display

expl_model = trainer.model.float().eval()

def explain(text):
    """Score each word: >0 pushes the prediction towards AI, <0 towards human.
    Gradient x Input = (how sensitive the AI-vs-human score is to each token's embedding) x (the embedding itself)."""
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0].tolist()
    word_ids = enc.word_ids()
    enc = {k: v.to(device) for k, v in enc.items()}
    embeds = expl_model.get_input_embeddings()(enc["input_ids"]).detach().requires_grad_(True)
    logits = expl_model(inputs_embeds=embeds, attention_mask=enc["attention_mask"]).logits[0]
    score = logits[1] - logits[0]                      # AI score minus human score
    score.backward()
    token_sal = (embeds.grad[0] * embeds[0]).sum(-1).detach().cpu().numpy()
    prob_ai = torch.softmax(logits.detach(), -1)[1].item()

    words = {}                                         # merge WordPieces back into words
    for tok_i, w_id in enumerate(word_ids):
        if w_id is None:                               # [CLS], [SEP]
            continue
        start, end = offsets[tok_i]
        if w_id not in words:
            words[w_id] = [start, end, 0.0]
        words[w_id][1] = end
        words[w_id][2] += float(token_sal[tok_i])
    result, prev_end = [], None
    for s, e, v in words.values():                     # (word, score, is there a space before it?)
        result.append((text[s:e], v, prev_end is not None and s > prev_end))
        prev_end = e
    return result, prob_ai

def show_html(word_scores, prob_ai, title):
    m = max(abs(v) for _, v, _ in word_scores) or 1
    html = ""
    for w, v, space in word_scores:
        rgb = (235, 104, 52) if v > 0 else (42, 120, 214)
        html += (" " if space else "") + (f'<span style="background: rgba({rgb[0]},{rgb[1]},{rgb[2]},{abs(v)/m:.2f}); '
                                          f'padding:1px 1px; border-radius:3px">{w}</span>')
    display(HTML(f"<b>{title}</b> - P(AI) = {prob_ai:.3f}<br><div style='line-height:1.9'>{html}</div>"))

def save_highlight_png(word_scores, prob_ai, title, path, max_words=110):
    """Same highlighted text, drawn with matplotlib so it can go into the report/slides."""
    from matplotlib.transforms import Bbox
    word_scores = word_scores[:max_words]
    m = max(abs(v) for _, v, _ in word_scores) or 1
    fig = plt.figure(figsize=(10, 4.2)); ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    renderer = fig.canvas.get_renderer()
    x, y, line_h = 0.02, 0.86, 0.075
    ax.text(0.02, 0.95, f"{title}   -   P(AI) = {prob_ai:.3f}   (orange = pushes to AI, blue = pushes to human)",
            fontsize=10, fontweight="bold", transform=ax.transAxes)
    for w, v, space in word_scores:
        color = AI_COLOR if v > 0 else HUMAN_COLOR
        if space:
            x += 0.007
        t = ax.text(x, y, w, fontsize=10, transform=ax.transAxes,
                    bbox=dict(boxstyle="round,pad=0.15", fc=color, alpha=min(1, abs(v) / m) * 0.85, ec="none"))
        width = t.get_window_extent(renderer).width / fig.bbox.width
        if x + width > 0.98:                         # wrap to next line
            t.remove(); x, y = 0.02, y - line_h
            t = ax.text(x, y, w, fontsize=10, transform=ax.transAxes,
                        bbox=dict(boxstyle="round,pad=0.15", fc=color, alpha=min(1, abs(v) / m) * 0.85, ec="none"))
        x += width + 0.001
    # crop the image to the text actually drawn
    box = Bbox.union([t.get_window_extent(renderer) for t in ax.texts]).transformed(fig.dpi_scale_trans.inverted())
    fig.savefig(path, bbox_inches=box.padded(0.15)); plt.close(fig)

# Pick one confidently-correct AI text and one human text of moderate length
n_words = test_df["text"].str.split().str.len()
ok_len = (n_words > 40) & (n_words < 110)
def pick_example(label):
    """Most confident correct prediction of this class (falls back to any text of the class)."""
    cands = test_df[(test_df["label"] == label) & (test_df["pred"] == label) & ok_len]
    if cands.empty:
        cands = test_df[test_df["label"] == label]
    cands = cands.sort_values("prob_ai")
    return cands.iloc[-1] if label == 1 else cands.iloc[0]

ai_example, hu_example = pick_example(1), pick_example(0)

for ex, name in [(ai_example, "AI-written example"), (hu_example, "Human-written example")]:
    ws, p = explain(ex["text"])
    show_html(ws, p, f"{name} ({ex['domain']})")
    save_highlight_png(ws, p, f"{name} ({ex['domain']})", FIG_DIR / f"saliency_{'ai' if ex['label'] else 'human'}_example.png")

**What this does:** answers *"why did BERT decide this?"* for a single text. **Gradient × Input** measures, for each word, how much the
AI-vs-human score would change if that word's embedding changed (the gradient), multiplied by the embedding itself.
WordPiece pieces are added back together into whole words. **Orange** words pushed the model towards *AI*, **blue** words towards *human*;
stronger colour = stronger influence. The same function powers the highlighting in our live demo.

**Concepts:** explainable AI (XAI) — the "Explainable" in our project title; **Unit 2: word embeddings** (the saliency is computed on each
token's embedding vector). **Unit 5: ethics** — a teacher should be able to see *why* a text was flagged before acting on it.

In [ ]:
# Step 15b - Which words push BERT towards AI or human overall? (saliency over many test texts)
from collections import defaultdict

n_global = min(300, len(test_df))
word_totals, word_counts = defaultdict(float), defaultdict(int)
for text in test_df["text"].sample(n_global, random_state=SEED):
    ws, _ = explain(text)
    for w, v, _ in ws:
        w = w.lower().strip(".,;:!?\"'()")
        if w.isalpha():
            word_totals[w] += v; word_counts[w] += 1

min_count = 3 if SMOKE_TEST else 8
avg = pd.Series({w: word_totals[w] / word_counts[w] for w in word_totals if word_counts[w] >= min_count})
top_ai, top_human = avg.sort_values(ascending=False).head(15), avg.sort_values().head(15)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].barh(top_human.index[::-1], -top_human.values[::-1], color=HUMAN_COLOR); axes[0].set_title("Words pushing to HUMAN")
axes[1].barh(top_ai.index[::-1], top_ai.values[::-1], color=AI_COLOR); axes[1].set_title("Words pushing to AI")
for a in axes: a.set_xlabel("average Gradient x Input")
fig.suptitle(f"BERT global word importance ({n_global} test texts, words seen >= {min_count} times)", fontweight="bold")
fig.tight_layout(); fig.savefig(FIG_DIR / "bert_global_saliency_words.png"); plt.show()
global_saliency = {"AI": [(w, round(v, 4)) for w, v in top_ai.items()],
                   "human": [(w, round(v, 4)) for w, v in top_human.items()]}

**What this does:** repeats the word-level explanation on a few hundred test texts and averages the scores per word, giving a
**global** picture of what BERT has learned. We can compare it to the top TF-IDF words of the Logistic Regression baseline from Phase 1.
If both point at similar words (e.g. formal, generic vocabulary for AI; concrete details for human), that supports our explanation.

**Syllabus link — Unit 2/4: comparing contextual (BERT) and frequency-based (TF-IDF) features.**

In [ ]:
# Step 16 - Last-layer attention analysis (with an important caveat)
att_model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR), attn_implementation="eager").to(device).eval()

def last_layer_attention(text):
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    with torch.no_grad():
        out = att_model(**enc, output_attentions=True)
    att = out.attentions[-1][0].mean(0).cpu().numpy()    # last layer, averaged over the attention heads
    return tokenizer.convert_ids_to_tokens(enc["input_ids"][0]), att

# (a) Heat-map for the start of the AI example
tokens, att = last_layer_attention(ai_example["text"])
n = min(28, len(tokens))
fig, ax = plt.subplots(figsize=(8.5, 7.5))
ax.imshow(att[:n, :n], cmap="Oranges"); ax.grid(False)
ax.set_xticks(range(n), tokens[:n], rotation=90, fontsize=8); ax.set_yticks(range(n), tokens[:n], fontsize=8)
ax.set_xlabel("attended-to token"); ax.set_ylabel("token doing the attending")
ax.set_title("Last-layer self-attention (mean of heads), first tokens of the AI example")
fig.savefig(FIG_DIR / "bert_attention_heatmap.png"); plt.show()

# (b) Where does [CLS] look?  Compare top-attention tokens with top-saliency words
cls_att = att[0]
top_att = [tokens[i] for i in np.argsort(cls_att)[::-1][:10]]
ws, _ = explain(ai_example["text"])
top_sal = [w for w, _, _ in sorted(ws, key=lambda t: -abs(t[1]))[:10]]
print("Top-10 tokens by [CLS] attention :", top_att)
print("Top-10 words by |saliency|       :", top_sal)

# (c) Over many texts: how much [CLS] attention goes to special tokens and punctuation?
share = {"[CLS]": [], "[SEP]": [], "punctuation": [], "words": []}
for text in test_df["text"].head(min(200, len(test_df))):
    toks, a = last_layer_attention(text)
    row = a[0]
    cls_ = row[[i for i, t in enumerate(toks) if t == "[CLS]"]].sum()
    sep_ = row[[i for i, t in enumerate(toks) if t == "[SEP]"]].sum()
    pun_ = row[[i for i, t in enumerate(toks) if all(not c.isalnum() for c in t) and t not in ("[CLS]", "[SEP]")]].sum()
    share["[CLS]"].append(cls_); share["[SEP]"].append(sep_); share["punctuation"].append(pun_)
    share["words"].append(1 - cls_ - sep_ - pun_)
attention_share = {k: round(float(np.mean(v)), 4) for k, v in share.items()}
print("Average share of [CLS] attention:", attention_share)

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(list(attention_share), list(attention_share.values()), color=[SERIES[6], SERIES[6], SERIES[3], SERIES[0]])
ax.bar_label(bars, fmt="%.2f", fontsize=9, color=MUTED)
ax.set_ylabel("share of [CLS] attention"); ax.set_title("Where the last layer's [CLS] attention goes")
fig.savefig(FIG_DIR / "bert_attention_share.png"); plt.show()

**What this does:** looks inside BERT's **self-attention** in the last layer. The heat-map shows, for each token (row), how much it
attends to every other token (column), averaged over the 12 attention heads. We then check where the **`[CLS]`** token — whose vector makes
the final decision — puts its attention, and how that compares to the Gradient × Input words.

**Important caveat — "attention is not explanation" (Jain & Wallace, 2019; see also Wiegreffe & Pinter, 2019):**
attention weights show *where the model looks*, not *why it decides*. Different attention patterns can give the same prediction, and a large
share of attention often goes to `[SEP]`, `[CLS]` and punctuation, which act as "parking spots" rather than evidence (Clark et al., 2019).
Panel (c) checks this on our model. That is why DetectAI uses **gradient-based saliency** as its main explanation and shows attention only
as a look inside the Transformer.

**Syllabus link — Unit 4: Transformers and attention** (multi-head self-attention, the `[CLS]` token), **Unit 5: responsible/ethical NLP.**

In [ ]:
# Step 17 - Error analysis: what does BERT get wrong?
test_df["n_words"] = test_df["text"].str.split().str.len()
errors = test_df[test_df["pred"] != test_df["label"]].copy()
errors["error_type"] = np.where(errors["label"] == 0, "False positive (human flagged as AI)", "False negative (AI missed)")
print(f"{len(errors)} of {len(test_df)} test texts misclassified\n")
print(pd.crosstab(errors["domain"], errors["error_type"], margins=True).to_string(), "\n")

pd.set_option("display.max_colwidth", 220)
for etype in ["False positive (human flagged as AI)", "False negative (AI missed)"]:
    sub = errors[errors["error_type"] == etype]
    sub = sub.sort_values("prob_ai", ascending=(etype.startswith("False negative")))
    print(f"--- Most confident {etype.lower()} ---")
    for _, r in sub.head(4).iterrows():
        print(f"[{r['domain']}, P(AI)={r['prob_ai']:.3f}, {r['n_words']} words] {r['text'][:220]}...\n")

# Accuracy vs text length
bins = [19, 60, 120, 200, 300]
test_df["length_bucket"] = pd.cut(test_df["n_words"], bins=bins, labels=["20-60", "61-120", "121-200", "201-300"])
acc_by_len = test_df.groupby("length_bucket", observed=False).apply(
    lambda d: accuracy_score(d["label"], d["pred"]) if len(d) else np.nan)
fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(acc_by_len.index.astype(str), acc_by_len.values, color=SERIES[5])
ax.bar_label(bars, fmt="%.3f", fontsize=9, color=MUTED)
ax.set_ylim(0.5, 1.02); ax.set_xlabel("Words per text"); ax.set_ylabel("Accuracy")
ax.set_title("BERT accuracy by text length (test)")
fig.savefig(FIG_DIR / "bert_accuracy_by_length.png"); plt.show()

errors[["domain", "error_type", "prob_ai", "n_words", "text"]].to_csv(OUT_DIR / "misclassified_examples.csv", index=False)

**What this does:** lists the texts BERT got wrong, split into two kinds of error:
- **False positive** — a *human* text flagged as AI. In a classroom this means wrongly accusing a student, so it is the more harmful error.
- **False negative** — an AI text that slipped through.

We print the most confident mistakes so we can read them and see *why* (e.g. very short texts, formal human writing that sounds "AI-like",
or casual AI answers), and plot accuracy by text length — short texts carry less evidence, so they are expected to be harder.
All errors are saved to `misclassified_examples.csv` for the report.

**Syllabus link — Unit 5: ethical issues and bias in NLP.** Formal or non-native writers may be flagged more often, so a detector should
support a human decision, never replace it.

In [ ]:
# Step 18 - Save all results and download everything as one zip file
bert_metrics = {
    "model": MODEL_NAME, "smoke_test": SMOKE_TEST,
    "hyperparameters": {"max_len": MAX_LEN, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
                        "max_epochs": MAX_EPOCHS, "warmup_ratio": WARMUP, "weight_decay": WEIGHT_DECAY,
                        "fp16": torch.cuda.is_available(), "early_stopping_patience": 1, "seed": SEED},
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    "train_minutes": round(train_minutes, 1), "best_epoch": best_epoch,
    "epoch_history": epoch_table.to_dict(orient="records"),
    "test": bert_test, "robustness": bert_robust,
    "global_saliency_words": global_saliency, "attention_share_cls": attention_share,
    "n_misclassified": int(len(errors)),
}
(OUT_DIR / "bert_metrics.json").write_text(json.dumps(bert_metrics, indent=2))

archive = shutil.make_archive("detectai_bert_outputs", "zip", root_dir=".", base_dir=str(OUT_DIR))
print(f"Created {archive} ({os.path.getsize(archive) / 1e6:.0f} MB)")

# A single ~400 MB browser download tends to freeze, so we cut the zip into 80 MB parts.
import glob, subprocess
PART_PREFIX = "detectai_bert_outputs.zip.part"
for old in glob.glob(PART_PREFIX + "*"):
    os.remove(old)
if shutil.which("split"):                      # Colab (Linux): split -b 80M -d
    subprocess.run(["split", "-b", "80M", "-d", archive, PART_PREFIX], check=True)
else:                                          # fallback for computers without `split`
    with open(archive, "rb") as f:
        i = 0
        while chunk := f.read(80 * 1024 * 1024):
            Path(f"{PART_PREFIX}{i:02d}").write_bytes(chunk)
            i += 1
parts = sorted(glob.glob(PART_PREFIX + "*"))
for part in parts:
    print(f"  {part}  ({os.path.getsize(part) / 1e6:.0f} MB)")

print("\nAfter downloading, put all parts in one folder on Windows, open Command Prompt there and run:\n")
print("    copy /b " + " + ".join(parts) + " detectai_bert_outputs.zip\n")

try:
    from google.colab import files
    files.download(str(OUT_DIR / "bert_metrics.json"))   # small file first, as a quick backup
    for part in parts:
        time.sleep(3)                          # give the browser time between downloads
        files.download(part)
except ImportError:
    pass

**What this does:** writes every number from this notebook into `bert_metrics.json` (test metrics, per-domain accuracy, robustness,
training history, top saliency words, attention shares) and zips it together with the **model, tokenizer and all figures** into
`detectai_bert_outputs.zip` (~400 MB, mostly the model weights).

Because one 400 MB browser download tends to freeze, the zip is **cut into 80 MB parts** (`split -b 80M -d`), named
`detectai_bert_outputs.zip.part00`, `part01`, … Each part downloads separately — if Chrome asks *"Download multiple files?"*, click **Allow**.
(`bert_metrics.json` also downloads on its own first, as a small backup.)

**On Windows**, put all parts in one folder, open *Command Prompt* in that folder and run the `copy /b …` command printed above —
it glues the parts back into `detectai_bert_outputs.zip`. Then unzip it.

If a download fails, open the 📁 *Files* panel on the left and right-click the missing part → *Download*.

**Next step:** put the model folder at `DETECTAI/backend/models/detectai-bert/`, the figures in `results/figures/` and
`bert_metrics.json` in `results/`, then run `scripts/calibrate.py` and restart `run.bat`.

## Summary — how this notebook covers the syllabus
| Unit | Topic | Step |
|---|---|---|
| 1 | Tokenization, morphology (WordPiece sub-words), ambiguity, NLP libraries (Hugging Face, NLTK) | 1, 6, 14 |
| 2 | Contextual embeddings (BERT) vs Bag of Words / TF-IDF, word embeddings, WordNet synonyms | 7, 8, 12, 14, 15 |
| 4 | Transformers & self-attention, Naive Bayes / LR / SVM comparison, accuracy / precision / recall / F1 / confusion matrix / ROC | 8, 11, 12, 16 |
| 5 | Applications (multi-domain detection), ethics & bias (false positives, robustness, explainability) | 13, 14, 15, 17 |